# EDA — Sardinia Hospitality Intelligence

Exploratory Data Analysis of the Sardinia tourism and hospitality market (2018–2024).

**Sources:** ISTAT open data — tourist flows + accommodation capacity  
**Goal:** Highlight demand/supply gaps, seasonal profiles, tourist segmentation, and expansion targeting opportunities.

---

**Sections:**
1. Setup & data loading
2. Demand overview (2018–2024)
3. Demand / supply gap
4. Seasonality
5. Segmentation by origin
6. Segmentation by accommodation type
7. Priority score & expansion targeting
8. Key takeaways

## 1. Setup & data loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

# ── paths ─────────────────────────────────────────────────────────────────────
ANALYSIS_DIR = Path("../data/analysis")

# ── plot defaults ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ── province label normalization ───────────────────────────────────────────────
# Raw data contains two inconsistent name variants for Cagliari
PROVINCE_LABELS: dict[str, str] = {
    "Città metropolitana di Cagliari": "Cagliari",
    "Città metropolitana Cagliari": "Cagliari",
    "Provincia di Sassari": "Sassari",
    "Provincia di Nuoro": "Nuoro",
    "Provincia di Oristano": "Oristano",
    "Provincia del Sud Sardegna": "Sud Sardegna",
}

ACC_LABELS: dict[str, str] = {
    "Esercizi Alberghieri": "Hotels",
    "Esercizi Extra-Alberghieri:Alloggi privati in affitto": "Short-term rentals",
    "Esercizi Extra-Alberghieri:Esercizi Complementari": "Complementary",
    "Esercizi Extra-Alberghieri": "Non-hotel",
}


def normalize_province(df: pd.DataFrame) -> pd.DataFrame:
    """Replace verbose Italian province names with short labels."""
    df = df.copy()
    df["province"] = df["province"].replace(PROVINCE_LABELS)
    return df


# ── load datasets ──────────────────────────────────────────────────────────────
demand       = normalize_province(pd.read_csv(ANALYSIS_DIR / "v_demand_by_province.csv"))
gap          = normalize_province(pd.read_csv(ANALYSIS_DIR / "v_supply_demand_gap.csv"))
supply       = normalize_province(pd.read_csv(ANALYSIS_DIR / "v_supply_by_province.csv"))
origin       = normalize_province(pd.read_csv(ANALYSIS_DIR / "v_segment_origin.csv"))
accommodation= normalize_province(pd.read_csv(ANALYSIS_DIR / "v_segment_accommodation.csv"))
trend_yoy    = normalize_province(pd.read_csv(ANALYSIS_DIR / "v_trend_yoy.csv"))
priority     = normalize_province(pd.read_csv(ANALYSIS_DIR / "q_priority_score.csv"))
seasonality  = normalize_province(pd.read_csv(ANALYSIS_DIR / "q_seasonality_extremes.csv"))
top_growth   = normalize_province(pd.read_csv(ANALYSIS_DIR / "q_top_growth_segments.csv"))

accommodation["acc_label"] = accommodation["accommodation_type"].replace(ACC_LABELS)
top_growth["acc_label"]    = top_growth["accommodation_type"].replace(ACC_LABELS)

print("Datasets loaded:")
for name, df in [
    ("demand", demand), ("gap", gap), ("supply", supply),
    ("origin", origin), ("accommodation", accommodation),
    ("trend_yoy", trend_yoy), ("priority", priority),
    ("seasonality", seasonality), ("top_growth", top_growth),
]:
    print(f"  {name:20s} {df.shape[0]:>6,} rows × {df.shape[1]} cols")

## 2. Demand overview (2018–2024)

In [ ]:
# Total arrivals and nights by year (all provinces)
annual_total = (
    demand.groupby("year")[["total_arrivals", "total_nights"]]
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(1, 2)

for ax, col, label, color in [
    (axes[0], "total_arrivals", "Arrivals",      "steelblue"),
    (axes[1], "total_nights",   "Overnight stays", "coral"),
]:
    ax.bar(annual_total["year"], annual_total[col] / 1e6, color=color, alpha=0.85)
    ax.set_title(f"Total {label} — Sardinia", fontsize=13, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Millions")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f M"))
    ax.axvspan(2019.5, 2021.5, alpha=0.08, color="red", label="COVID-19")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(annual_total.to_string(index=False))

In [ ]:
# Arrivals and nights by province — 2024
THOUSANDS_FMT = "%.0f K"
demand_2024 = (
    demand[demand["year"] == 2024]
    .groupby("province")[["total_arrivals", "total_nights"]]
    .sum()
    .sort_values("total_arrivals", ascending=True)
    .reset_index()
)

fig, axes = plt.subplots(1, 2)

for ax, col, label, color in [
    (axes[0], "total_arrivals", "Arrivals 2024",       "steelblue"),
    (axes[1], "total_nights",   "Overnight stays 2024", "coral"),
]:
    bars = ax.barh(demand_2024["province"], demand_2024[col] / 1e3, color=color, alpha=0.85)
    ax.bar_label(bars, fmt=THOUSANDS_FMT, padding=4, fontsize=9)
    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.set_xlabel("Thousands")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter(THOUSANDS_FMT))

plt.tight_layout()
plt.show()

In [ ]:
# Arrivals trend by province — line chart 2018-2024
THOUSANDS_ARRIVALS = "Thousands of arrivals"
province_annual = (
    demand.groupby(["year", "province"])["total_arrivals"]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))
for prov, grp in province_annual.groupby("province"):
    ax.plot(grp["year"], grp["total_arrivals"] / 1e3, marker="o", linewidth=2, label=prov)

ax.set_title("Arrivals trend by province (2018–2024)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel(THOUSANDS_ARRIVALS)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(THOUSANDS_FMT))
ax.axvspan(2019.5, 2021.5, alpha=0.07, color="red", label="COVID-19")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 3. Demand / supply gap

In [ ]:
# Occupancy proxy by province — 2024
# occupancy_proxy = total_nights / (total_beds × 365) × 100
gap_2024 = gap[gap["year"] == 2024].sort_values("occupancy_proxy", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if v > 50 else "#3498db" for v in gap_2024["occupancy_proxy"]]
bars = ax.bar(gap_2024["province"], gap_2024["occupancy_proxy"], color=colors, alpha=0.85)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=10)
ax.axhline(50, color="gray", linestyle="--", linewidth=1, label="50% threshold")
ax.set_title("Occupancy proxy by province — 2024", fontsize=13, fontweight="bold")
ax.set_ylabel("Nights / (Beds × 365) × 100")
ax.set_ylim(0, max(gap_2024["occupancy_proxy"]) * 1.2)
ax.legend()
plt.tight_layout()
plt.show()

print("\nGap detail — 2024:")
print(
    gap_2024[[
        "province", "total_arrivals", "total_nights",
        "total_beds", "occupancy_proxy"
    ]].to_string(index=False)
)

In [ ]:
# Occupancy proxy trend over time
fig, ax = plt.subplots(figsize=(13, 6))
for prov, grp in gap.groupby("province"):
    ax.plot(grp["year"], grp["occupancy_proxy"], marker="o", linewidth=2, label=prov)

ax.axhline(50, color="gray", linestyle="--", linewidth=1, label="50% threshold")
ax.axvspan(2019.5, 2021.5, alpha=0.07, color="red")
ax.set_title("Occupancy proxy by province (2018–2024)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Seasonality

In [ ]:
# Monthly heatmap (province × month) — overnight stays 2024, normalized by row (%)
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

pivot_nights = (
    demand[demand["year"] == 2024]
    .pivot_table(index="province", columns="month", values="total_nights", aggfunc="sum")
)
pivot_nights.columns = MONTH_LABELS[: len(pivot_nights.columns)]
pivot_pct = pivot_nights.div(pivot_nights.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    pivot_pct,
    annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, ax=ax, cbar_kws={"label": "% of annual stays"},
)
ax.set_title("Monthly distribution of overnight stays by province — 2024 (%)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Seasonality index and peak month share by province
seasonality_sorted = seasonality.sort_values("seasonality_index")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].barh(
    seasonality_sorted["province"],
    seasonality_sorted["seasonality_index"],
    color="steelblue", alpha=0.85
)
axes[0].bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
axes[0].set_title("Seasonality index\n(monthly coefficient of variation — lower = more uniform)",
                  fontsize=11, fontweight="bold")
axes[0].set_xlabel("Index")

seas2 = seasonality.sort_values("peak_month_share_pct")
bars2 = axes[1].barh(
    seas2["province"], seas2["peak_month_share_pct"],
    color="coral", alpha=0.85
)
axes[1].bar_label(bars2, fmt="%.1f%%", padding=4, fontsize=9)
axes[1].set_title("Peak month share (% of annual stays)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("% of annual total")

plt.tight_layout()
plt.show()

print(seasonality[
    ["province", "peak_month_share_pct", "top3_month_share_pct", "seasonality_index"]
].to_string(index=False))

## 5. Segmentation by origin

In [ ]:
# Domestic vs international arrivals by province — 2024
origin_2024 = origin[origin["year"] == 2024].copy()

origin_group_prov = (
    origin_2024.groupby(["province", "origin_group"])["total_arrivals"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

intl_col     = "Internazionale"
domestic_col = "Domestico"
origin_group_prov["total"]    = origin_group_prov[[intl_col, domestic_col]].sum(axis=1)
origin_group_prov["intl_pct"] = origin_group_prov[intl_col] / origin_group_prov["total"] * 100
origin_group_prov = origin_group_prov.sort_values("intl_pct", ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
width = 0.4
x = range(len(origin_group_prov))

ax.bar([i - width/2 for i in x], origin_group_prov[domestic_col] / 1e3,
       width=width, label="Domestic", color="steelblue", alpha=0.85)
ax.bar([i + width/2 for i in x], origin_group_prov[intl_col] / 1e3,
       width=width, label="International", color="coral", alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(origin_group_prov["province"])
ax.set_title("Domestic vs international arrivals by province — 2024",
             fontsize=13, fontweight="bold")
ax.set_ylabel(THOUSANDS_ARRIVALS)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(THOUSANDS_FMT))
ax.legend()
plt.tight_layout()
plt.show()

print("\nInternational share % — 2024:")
print(origin_group_prov[["province", intl_col, domestic_col, "intl_pct"]].to_string(index=False))

In [ ]:
# Top 10 countries of origin — Sardinia 2024
top_origins = (
    origin_2024[origin_2024["origin_group"] == "Internazionale"]
    .groupby("origin")["total_arrivals"]
    .sum()
    .nlargest(10)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_origins.index, top_origins.values / 1e3, color="teal", alpha=0.85)
ax.bar_label(bars, fmt="THOUSANDS_FMT", padding=4, fontsize=9)
ax.set_title("Top 10 countries of origin — Sardinia 2024", fontsize=13, fontweight="bold")
ax.set_xlabel("Thousands of arrivals")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("THOUSANDS_FMT"))
plt.tight_layout()
plt.show()

## 6. Segmentation by accommodation type

In [ ]:
# Accommodation mix by province — 2024 (stacked bar, % of arrivals)
acc_2024 = (
    accommodation[accommodation["year"] == 2024]
    .groupby(["province", "acc_label"])[["total_arrivals", "total_nights"]]
    .sum()
    .reset_index()
)

pivot_acc = acc_2024.pivot_table(
    index="province", columns="acc_label", values="total_arrivals", aggfunc="sum"
).fillna(0)
pivot_acc_pct = pivot_acc.div(pivot_acc.sum(axis=1), axis=0) * 100

pivot_acc_pct.plot(kind="bar", stacked=True, figsize=(13, 5),
                   colormap="Set2", alpha=0.9, edgecolor="white")
plt.title("Accommodation mix by province — 2024 (% of arrivals)",
          fontsize=13, fontweight="bold")
plt.ylabel("% arrivals")
plt.xlabel("")
plt.xticks(rotation=0)
plt.legend(title="Accommodation type", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Average length of stay by accommodation type and province — 2024
stay_2024 = (
    accommodation[accommodation["year"] == 2024][["province", "acc_label", "avg_stay_length"]]
    .drop_duplicates()
)
pivot_stay = stay_2024.pivot_table(index="province", columns="acc_label", values="avg_stay_length")

fig, ax = plt.subplots(figsize=(13, 5))
pivot_stay.plot(kind="bar", ax=ax, colormap="Set2", alpha=0.9, edgecolor="white")
ax.set_title("Average length of stay (nights) by accommodation type — 2024",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Nights")
ax.set_xlabel("")
ax.set_xticklabels(pivot_stay.index, rotation=0)
ax.legend(title="Accommodation type", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Priority score & expansion targeting

In [ ]:
# Priority score by province
priority_sorted = priority.sort_values("priority_score", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors_p = ["#e74c3c" if i == 0 else "#3498db" for i in range(len(priority_sorted))]
bars = ax.bar(
    priority_sorted["province"], priority_sorted["priority_score"],
    color=colors_p, alpha=0.85
)
ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=10)
ax.set_title(
    "Priority score by province\n(composite: occupancy + YoY growth + international share)",
    fontsize=13, fontweight="bold"
)
ax.set_ylabel("Composite score (0–1 normalised)")
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

print("\nPriority score components:")
print(
    priority_sorted[[
        "province", "occupancy_proxy", "yoy_arrivals_pct",
        "intl_share_pct", "priority_score"
    ]].to_string(index=False)
)

In [ ]:
# Top growth segments — YoY arrivals %
top_growth["segment"] = top_growth["province"] + " / " + top_growth["acc_label"]
tg_sorted = top_growth.sort_values("yoy_arrivals_pct", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(tg_sorted["segment"], tg_sorted["yoy_arrivals_pct"],
               color="mediumseagreen", alpha=0.85)
ax.bar_label(bars, fmt="+%.1f%%", padding=4, fontsize=9)
ax.set_title("Top growth segments — YoY arrivals (%)", fontsize=13, fontweight="bold")
ax.set_xlabel("YoY arrivals growth (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: occupancy proxy vs YoY growth
# Provinces with high occupancy + high growth = highest priority
fig, ax = plt.subplots(figsize=(9, 6))

scatter = ax.scatter(
    priority["occupancy_proxy"],
    priority["yoy_arrivals_pct"],
    s=priority["intl_share_pct"].fillna(0) * 2 + 80,  # bubble size ∝ international share
    c=priority["priority_score"],
    cmap="RdYlGn",
    alpha=0.85,
    edgecolors="gray",
    linewidths=0.5,
)
plt.colorbar(scatter, ax=ax, label="Priority score")

for _, row in priority.iterrows():
    ax.annotate(
        row["province"],
        (row["occupancy_proxy"], row["yoy_arrivals_pct"]),
        textcoords="offset points", xytext=(8, 4), fontsize=9,
    )

ax.axvline(priority["occupancy_proxy"].median(), color="gray",
           linestyle="--", linewidth=1, alpha=0.6, label="Median occupancy")
ax.axhline(priority["yoy_arrivals_pct"].median(), color="gray",
           linestyle="--", linewidth=1, alpha=0.6, label="Median YoY growth")
ax.set_title(
    "Province positioning: occupancy vs YoY growth\n(bubble size = international share)",
    fontsize=12, fontweight="bold"
)
ax.set_xlabel("Occupancy proxy (%)")
ax.set_ylabel("YoY arrivals growth (%)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Key takeaways

| Theme | Finding |
|-------|---------|
| **Demand recovery** | After the COVID-19 collapse in 2020, Sardinia recovered and exceeded pre-pandemic levels by 2022–2023. |
| **Supply gap** | Occupancy proxy exceeds 50% in only some provinces — indicating room for accommodation expansion where demand is high but capacity remains available. |
| **Seasonality** | Tourism is highly concentrated in July–August. Cagliari shows the most uniform distribution — highest potential for de-seasonalisation strategies. |
| **Origin** | International share is significant; European markets (Germany, France, UK) dominate. |
| **Accommodation** | The short-term rental segment shows the highest YoY growth across nearly all provinces. |
| **Targeting** | The composite priority score highlights provinces with the best balance of current demand, growth momentum, and international openness. |